# CEO EVSI con demora
Caso sintético que extiende el gate de pricing previo. Calcula valor exacto preposterior, compara decisión inmediata y piloto, y registra límites.

In [2]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path('src').resolve()))
from evsi_delay import Arm, analyze
cfg=json.loads(Path('config.json').read_text())
a=Arm(**cfg['prior']['arm_a']); b=Arm(**cfg['prior']['arm_b'])
print('synthetic:', cfg['synthetic'], '| n/arm:', cfg['n_per_arm'], '| H:', cfg['horizon_periods'])

synthetic: True | n/arm: 10 | H: 100


## Fórmula
Para margen posterior esperado q por periodo, margen inmediato m0, horizonte H, demora d y costo c: V_now=H*m0; V_pilot=(H-d)*q-c. Pilotear si la diferencia es estrictamente positiva. El break-even contra lanzar ahora es d*=H-(H*m0+c)/q.

In [4]:
rows=[analyze(a,b,cfg['n_per_arm'],cfg['horizon_periods'],d,cfg['pilot_cost']) for d in cfg['delay_periods']]
for r in rows:
    print(f"d={r['delay']:>3}  gain={r['net_gain']:>10.4f}  decision={r['decision']}  mass={r['predictive_mass']:.12f}")

d=  0  gain=   32.6331  decision=pilot  mass=1.000000000000
d=  2  gain=   20.9405  decision=pilot  mass=1.000000000000
d=  5  gain=    3.4015  decision=pilot  mass=1.000000000000
d= 10  gain=  -25.8302  decision=immediate  mass=1.000000000000
d= 15  gain=  -55.0618  decision=immediate  mass=1.000000000000
d= 20  gain=  -84.2935  decision=immediate  mass=1.000000000000
d= 30  gain= -142.7568  decision=immediate  mass=1.000000000000
d= 50  gain= -259.6834  decision=immediate  mass=1.000000000000
d=100  gain= -552.0000  decision=immediate  mass=1.000000000000


## Lectura
La suma enumera todos los conteos de ambos brazos y no usa Monte Carlo para la cifra exacta. El costo de demora es la pérdida de periodos del horizonte; el piloto no vende mientras aprende. La simulación independiente y las limitaciones se guardan en results/.

In [6]:
r=rows[0]; m0=max(a.margin*a.alpha/(a.alpha+a.beta), b.margin*b.alpha/(b.alpha+b.beta)); q=r['posterior_best_margin']; H=cfg['horizon_periods']; c=cfg['pilot_cost']
print('m0=',m0,'q=',q,'pilot-vs-launch break-even delay=',H-(H*m0+c)/q)
print('exact EVSI without delay=',r['evsi'],'roundoff mass error=',r['roundoff_mass_error'])

m0= 5.5 q= 5.846331238444178 pilot-vs-launch break-even delay= 5.5818123389639
exact EVSI without delay= 34.63312384441781 roundoff mass error= 2.1538326677728037e-14
